<div dir="rtl">

# خط لوله‌ی کامل پایان‌نامه: دسته‌بندی گزارش‌های حوادث دریایی MAIB

این نوت‌بوک کل پروژه را **مرحله به مرحله** اجرا می‌کند؛ از داده‌ی خام تا جدول نتایج و تفسیرپذیری.

| مرحله | کار | زمان تقریبی روی GTX 1650 Ti |
|---|---|---|
| ۰ | تنظیمات و بررسی GPU | — |
| ۱ | آماده‌سازی و تحلیل اکتشافی داده | < ۱ دقیقه |
| ۲ | مبنای کلاسیک TF-IDF | < ۱ دقیقه |
| ۳ | تست سریع (smoke) | ~۱۰ دقیقه |
| ۴ | آزمایش اصلی: ۳۶ اجرا | چند ساعت تا یک روز |
| ۵ | منحنی‌های آموزش | — |
| ۶ | جدول نتایج و آزمون معناداری | ~۲ دقیقه |
| ۷ | تحلیل خطا: ماتریس درهم‌ریختگی | — |
| ۸ | تفسیرپذیری با attention | ~۲۰ دقیقه |

**اصل طراحی:** منطق اصلی در `maib/` و اسکریپت‌های `.py` است و این نوت‌بوک فقط آن‌ها را صدا می‌زند و نتایج را نمایش می‌دهد.
پس نتیجه‌ی نوت‌بوک و نتیجه‌ی اجرای اسکریپت‌ها از ترمینال **دقیقاً یکی** است و کد در دو جا تکرار نشده.

> اگر فایلی داخل `maib/` را ویرایش کردی، از منوی Kernel گزینه‌ی **Restart** را بزن تا تغییر اعمال شود.

**قابلیت ادامه:** اجراهای تمام‌شده در `results/` ذخیره می‌شوند. اگر وسط مرحله‌ی ۴ لپ‌تاپ خاموش شد،
سلول‌های ۰ تا ۴ را دوباره اجرا کن؛ اجراهای قبلی رد می‌شوند و از همان‌جا ادامه می‌دهد.

</div>

<div dir="rtl">

## مرحله‌ی ۰: تنظیمات

**فقط همین سلول را تغییر بده.** بقیه‌ی نوت‌بوک از این متغیرها استفاده می‌کند.

</div>

In [ ]:
# ===================== تنظیمات =====================
MODEL      = "bert-base-uncased"
BATCH      = 8       # GTX 1650 Ti (4GB): 8 — اگر CUDA out of memory گرفتی: 4 و GRAD_ACCUM=4
GRAD_ACCUM = 2       # batch مؤثر = BATCH × GRAD_ACCUM = 16
DATA_FILE  = None    # اگر Hugging Face در دسترس نیست، مسیر فایل jsonl/csv را بده؛ مثلاً "maib.jsonl"
MIN_COUNT  = 20     # کلاس‌های کمتر از این تعداد گزارش حذف می‌شوند (۳ = نگه داشتن همه‌ی کلاس‌های ممکن)
EXTRA_ARGS = ""      # آرگومان‌های اضافه برای train.py؛ مثلاً "--max_len 256" یا "--grad_ckpt"

# کدام مراحلِ سنگین اجرا شوند
RUN_SMOKE    = True   # مرحله ۳
RUN_MAIN     = True   # مرحله ۴: ۳۶ اجرا
RUN_PAPER    = False  # مرحله ۴-ب: بازتولید مقاله‌ی مرجع با lr=1e-6
RUN_ABLATION = False  # مرحله ۴-ج: mean / attention مستقیم روی BERT
RUN_EXPLAIN  = True   # مرحله ۸

# پیکربندی‌ای که تفسیر می‌شود (روش پیشنهادی)
EXPLAIN_HEAD, EXPLAIN_LOSS, EXPLAIN_SEED = "bilstm_att", "focal", 1

In [ ]:
import json
import shlex
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, Markdown, display

assert Path("maib").is_dir(), "نوت‌بوک باید از داخل پوشه‌ی maib-thesis اجرا شود"

MODEL_SHORT = MODEL.rstrip("/").split("/")[-1]
COMMON = f"--models {MODEL} --batch {BATCH} --grad_accum {GRAD_ACCUM} {EXTRA_ARGS}"
pd.set_option("display.width", 200, "display.max_columns", 30)

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  |  VRAM: {p.total_memory / 2**30:.1f} GB  |  fp16 فعال")
else:
    print("⚠️ GPU پیدا نشد؛ آموزش BERT روی CPU بسیار کند است (مراحل ۱ و ۲ مشکلی ندارند).")
print("torch", torch.__version__, "| arguments:", COMMON)


def load_runs(results_dir):
    """همه‌ی فایل‌های نتیجه‌ی یک پوشه را می‌خواند."""
    return [json.loads(f.read_text()) for f in sorted(Path(results_dir).glob("*.json"))]

<div dir="rtl">

## مرحله‌ی ۱: آماده‌سازی و تحلیل اکتشافی داده

این سلول `prepare_data.py` را اجرا می‌کند:
1. **یکسان‌سازی برچسب‌ها:** برچسب‌هایی که فقط در حروف بزرگ و کوچک فرق دارند یکی می‌شوند.
2. **حذف تکراری‌ها:** اول تکراری‌هایی که برچسب متناقض دارند و بعد تکراری‌های عادی. بدون این کار ممکن است یک متن هم در train باشد و هم در test.
3. **تقسیم stratified به نسبت ۷۰/۱۵/۱۵** با seed ثابت ۴۲ در پوشه‌ی `splits/`. همه‌ی مدل‌ها روی همین تقسیم مقایسه می‌شوند.

بعد از اجرا، متغیرهای `df`، `dist` و `words` در نوت‌بوک در دسترس‌اند.

</div>

In [ ]:
DATA_ARG = (f"--data {DATA_FILE} " if DATA_FILE else "") + f"--min_count {MIN_COUNT}"
%run prepare_data.py $DATA_ARG

<div dir="rtl">

### توزیع کلاس‌ها: مشکل اصلی پایان‌نامه

به **نسبت عدم توازن** (بزرگ‌ترین کلاس تقسیم بر کوچک‌ترین) دقت کن. این عدد دلیل استفاده از macro-F1 به‌جای accuracy
و دلیل آزمودن توابع هزینه‌ی حساس به عدم توازن است. این نمودار مستقیم در فصل ۳ می‌رود.

</div>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1.4, 1]})

d = dist.sort_values()
axes[0].barh(d.index, d.values, color="#4C72B0")
for i, v in enumerate(d.values):
    axes[0].text(v, i, f" {v} ({100 * v / d.sum():.1f}%)", va="center", fontsize=8)
axes[0].set_title(f"Label distribution (imbalance ratio = {d.max() / d.min():.1f})")
axes[0].set_xlabel("reports")
axes[0].set_xlim(0, d.max() * 1.25)

axes[1].hist(words.clip(upper=words.quantile(0.99)), bins=50, color="#55A868")
axes[1].axvline(words.median(), color="k", ls="--", label=f"median = {words.median():.0f} words")
axes[1].set_title("Report length (words, clipped at 99th pct)")
axes[1].legend()
plt.tight_layout()
plt.show()

<div dir="rtl">

### بررسی سلامت تقسیم‌ها

نسبت هر کلاس باید در train، val و test تقریباً برابر باشد؛ معنی stratified همین است.

</div>

In [ ]:
from maib.data import load_splits

train_df, val_df, test_df, LABELS = load_splits("splits")
share = pd.DataFrame({name: d["label"].value_counts(normalize=True).mul(100).round(1)
                      for name, d in [("train %", train_df), ("val %", val_df), ("test %", test_df)]})
share["test n"] = test_df["label"].value_counts()
share.sort_values("train %")

<div dir="rtl">

> ⚠️ ستون `test n` را ببین. برای کلاسی که فقط چند نمونه در test دارد، یک پیش‌بینی درست یا غلط F1 را چند ده درصد جابه‌جا می‌کند.
> به همین دلیل نتایج را روی ۳ seed گزارش می‌کنیم و آزمون معناداری انجام می‌دهیم.

## مرحله‌ی ۲: مبنای کلاسیک TF-IDF

اگر یک مدل خطی ساده نزدیک به BERT عمل کند، این خودش یک یافته است. بدون این مبنا، ادعای برتری BERT قابل دفاع نیست.
خروجی‌ها با همان قالب BERT در `results/` ذخیره می‌شوند تا مرحله‌ی ۶ همه را کنار هم بگذارد.

</div>

In [ ]:
%run baseline.py

<div dir="rtl">

## مرحله‌ی ۳: تست سریع (smoke test)

یک اجرای کامل با **فقط یک epoch**. هدف دقت نیست؛ هدف این است که قبل از شروع کار چندساعته مطمئن شویم:
GPU، حافظه، توکنایزر، مدل و ذخیره‌ی نتیجه همگی بدون خطا کار می‌کنند.

اگر اینجا `CUDA out of memory` گرفتی، در مرحله‌ی ۰ مقدار `BATCH=4` و `GRAD_ACCUM=4` بگذار و دوباره امتحان کن.
خروجی در `results_smoke/` ذخیره می‌شود تا با نتایج اصلی قاطی نشود.

</div>

In [ ]:
if RUN_SMOKE:
    %run run_experiments.py --preset smoke $COMMON
    for r in load_runs("results_smoke"):
        print(f"{r['run']}: macro-F1={r['test']['macro_f1']:.3f}  ({r['train_seconds']:.0f}s)")
else:
    print("skipped (RUN_SMOKE=False)")

<div dir="rtl">

## مرحله‌ی ۴: آزمایش اصلی

حاصل‌ضرب **۳ head × ۴ loss × ۳ seed = ۳۶ اجرا**:

| head | نقش |
|---|---|
| `cls` | BERT ساده (مبنا) |
| `bilstm` | BERT + BiLSTM مقاله‌ی مرجع |
| `bilstm_att` | **روش پیشنهادی:** BiLSTM + توجه |

| loss | ایده |
|---|---|
| `ce` | بدون اقدام برای عدم توازن |
| `wce` | وزن معکوس فراوانی |
| `focal` | تمرکز روی نمونه‌های سخت |
| `cb` | وزن بر اساس «تعداد مؤثر نمونه» |

در هر اجرا: انتخاب بهترین epoch **فقط با val** (macro-F1) انجام می‌شود و ارزیابی روی test فقط یک‌بار در پایان است.

> این سلول طولانی است. اگر قطع شد، سلول‌های مرحله‌ی ۰ و همین سلول را دوباره اجرا کن؛ اجراهای تمام‌شده رد می‌شوند.

</div>

In [ ]:
if RUN_MAIN:
    %run run_experiments.py --preset main $COMMON
else:
    print("skipped (RUN_MAIN=False)")

<div dir="rtl">

### ۴-ب و ۴-ج: آزمایش‌های تکمیلی (اختیاری)

- **paper:** بازتولید با تنظیمات دقیق مقاله‌ی مرجع (lr=1e-6 و ۳۰ epoch). نتایج در `results_paper/` ذخیره می‌شود.
  batch مؤثر مقاله ۳۲ است، پس اینجا `GRAD_ACCUM` دو برابر می‌شود.
- **ablation:** آیا BiLSTM لازم است، یا توجه مستقیم روی BERT کافی است؟ نتایج در `results/` ذخیره می‌شود و در جدول اصلی می‌آید.

</div>

In [ ]:
if RUN_PAPER:
    PAPER_ACCUM = GRAD_ACCUM * 2
    %run run_experiments.py --preset paper --models $MODEL --batch $BATCH --grad_accum $PAPER_ACCUM $EXTRA_ARGS
if RUN_ABLATION:
    %run run_experiments.py --preset ablation $COMMON
if not (RUN_PAPER or RUN_ABLATION):
    print("skipped (RUN_PAPER=False, RUN_ABLATION=False)")

<div dir="rtl">

## مرحله‌ی ۵: منحنی‌های آموزش

macro-F1 روی val در هر epoch، میانگین روی seedها. چیزهایی که باید دنبالشان بگردی:
- **منحنی‌ای که هنوز رو به بالاست:** تعداد epoch کم بوده است.
- **افت بعد از epoch اول یا دوم:** overfitting. early stopping باید آن را گرفته باشد.
- **lossی که مدام صفر می‌ماند:** معمولاً یعنی مدل فقط کلاس پرتکرار را پیش‌بینی می‌کند.

</div>

In [ ]:
bert_runs = [r for r in load_runs("results") if r.get("history")]
if not bert_runs:
    print("هنوز اجرای BERT در results/ نیست؛ اول مرحله‌ی ۴ را اجرا کن.")
else:
    rows = [{"head": r["config"]["head"], "loss": r["config"]["loss"], **h}
            for r in bert_runs for h in r["history"]]
    hist = pd.DataFrame(rows).groupby(["head", "loss", "epoch"], as_index=False)["val_macro_f1"].mean()
    heads = sorted(hist["head"].unique())
    fig, axes = plt.subplots(1, len(heads), figsize=(5 * len(heads), 3.8), sharey=True, squeeze=False)
    for ax, head in zip(axes[0], heads):
        for loss, g in hist[hist["head"] == head].groupby("loss"):
            ax.plot(g["epoch"], g["val_macro_f1"], marker="o", label=loss)
        ax.set_title(head)
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.3)
    axes[0][0].set_ylabel("val macro-F1 (mean over seeds)")
    axes[0][-1].legend(title="loss")
    plt.tight_layout()
    plt.show()

<div dir="rtl">

## مرحله‌ی ۶: جدول نتایج و آزمون معناداری

این سلول `aggregate.py` را اجرا می‌کند که سه کار انجام می‌دهد:
1. **میانگین ± انحراف معیار** روی seedها برای هر پیکربندی. این همان جدول اصلی فصل ۴ است.
2. **Critical recall:** میانگین recall روی کلاس‌های نادر و ایمنی‌بحرانی (آتش‌سوزی، واژگونی، بدنه و آب‌گرفتگی).
3. **bootstrap جفت‌شده** در برابر پیکربندی مرجع، یعنی بازتولید مقاله (`bilstm/ce`). جواب این سؤال است که آیا بهبود واقعی است یا شانسی.

خروجی‌ها در `reports/` ذخیره می‌شوند.

</div>

In [ ]:
REFERENCE = f"{MODEL_SHORT}/bilstm/ce"
%run aggregate.py --reference $REFERENCE

<div dir="rtl">

### F1 هر کلاس (نادرترین کلاس در سمت چپ)

ستون‌های سمت چپ همان جایی‌اند که روش‌ها واقعاً از هم جدا می‌شوند. اگر روش پیشنهادی فقط در ستون‌های راست بهتر باشد، ادعای «بهبود روی رده‌های نادر» پشتوانه ندارد.

</div>

In [ ]:
pc = pd.read_csv("reports/per_class_f1.csv", index_col="config")
fig, ax = plt.subplots(figsize=(1.0 * pc.shape[1] + 3, 0.45 * len(pc) + 1.5))
im = ax.imshow(pc.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(pc.shape[1]), pc.columns, rotation=40, ha="right")
ax.set_yticks(range(len(pc)), pc.index)
for i in range(pc.shape[0]):
    for j in range(pc.shape[1]):
        ax.text(j, i, f"{pc.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("Per-class F1 (rows sorted by macro-F1, rarest class first)")
fig.colorbar(im, ax=ax, fraction=0.02)
plt.tight_layout()
plt.show()

<div dir="rtl">

### خوانش آزمون معناداری

`delta`: میانگین اختلاف macro-F1 با مرجع. `max_p`: بدترین p-value در بین seedها.
یک پیکربندی را فقط وقتی «به‌طور معنادار بهتر» بنام که `delta > 0` باشد **و** `max_p < 0.05` باشد؛ یعنی در **همه‌ی** seedها.

> ⚠️ ده‌ها مقایسه انجام شده است. با سطح ۰٫۰۵، چند نتیجه‌ی «معنادار» ممکن است تصادفی باشد.
> پیشنهاد می‌شود در پایان‌نامه اصلاح Holm-Bonferroni را هم گزارش کنی.

</div>

In [ ]:
sig_file = Path("reports/significance.csv")
if sig_file.exists():
    sig = pd.read_csv(sig_file).groupby("config").agg(
        delta=("delta_macro_f1", "mean"), max_p=("p_value", "max"), seeds=("seed", "count"))
    sig["significant (all seeds p<0.05)"] = (sig["delta"] > 0) & (sig["max_p"] < 0.05)
    display(sig.sort_values("delta", ascending=False).round(4))
else:
    print(f"مرجع {REFERENCE} پیدا نشد؛ آزمون معناداری انجام نشد.")

<div dir="rtl">

## مرحله‌ی ۷: تحلیل خطا با ماتریس درهم‌ریختگی

برای بهترین پیکربندی BERT، ماتریس‌های درهم‌ریختگی همه‌ی seedها جمع و **سطری نرمال** می‌شوند.
پس قطر اصلی برابر recall هر کلاس است. خانه‌های پررنگ خارج از قطر نشان می‌دهند کدام کلاس‌ها با هم اشتباه گرفته می‌شوند.
این برای بخش «تحلیل خطا» در فصل ۴ است. مثال سؤالی که باید جواب بدهی: آیا خطاها از نظر معنایی منطقی‌اند؟

</div>

In [ ]:
summary = pd.read_csv("reports/summary.csv")
bert_cfgs = [c for c in summary["config"] if not c.startswith("tfidf")]
if not bert_cfgs:
    print("هنوز نتیجه‌ی BERT نیست.")
else:
    best = bert_cfgs[0]  # summary بر اساس macro-F1 مرتب شده
    m, h, l = best.split("/")
    rs = [r for r in load_runs("results") if r["config"].get("head") == h and r["config"]["loss"] == l
          and r["config"]["model"].rstrip("/").split("/")[-1] == m]
    cm = np.sum([np.array(r["test"]["confusion_matrix"]) for r in rs], axis=0)
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    labels = rs[0]["labels"]

    fig, ax = plt.subplots(figsize=(9, 7.5))
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            if cm[i, j]:
                ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=6.5,
                        color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(f"Confusion matrix — {best} (summed over {len(rs)} seeds, row-normalised)")
    plt.tight_layout()
    plt.show()

    off = pd.DataFrame(cm, index=labels, columns=labels)
    pairs = [(off.index[i], off.columns[j], off.iat[i, j])
             for i in range(len(labels)) for j in range(len(labels)) if i != j and off.iat[i, j] > 0]
    display(pd.DataFrame(sorted(pairs, key=lambda x: -x[2])[:8], columns=["true", "predicted", "count"]))

<div dir="rtl">

## مرحله‌ی ۸: تفسیرپذیری با وزن‌های attention

مرحله‌ی ۴ برای صرفه‌جویی در فضای دیسک checkpointها را پاک می‌کند (هرکدام حدود ۴۴۰ مگابایت).
پس اینجا روش پیشنهادی **یک‌بار دیگر** با `keep_ckpt` آموزش داده می‌شود. نتیجه در `results_explain/` ذخیره می‌شود تا جدول اصلی دست نخورد.
بعد `explain_attention.py` وزن توجه زیرواژه‌ها را به سطح کلمه جمع می‌زند.

> ⚠️ **محدودیت علمی:** وزن attention لزوماً *علت* تصمیم مدل نیست (Jain & Wallace, 2019).
> در پایان‌نامه بنویس «کلماتی که مدل بیشترین وزن را به آن‌ها داده»، نه «دلیل تصمیم مدل».

</div>

In [ ]:
from train import get_parser, run, run_name

if RUN_EXPLAIN:
    cfg, _ = get_parser().parse_known_args(
        ["--model", MODEL, "--batch", str(BATCH), "--grad_accum", str(GRAD_ACCUM), *shlex.split(EXTRA_ARGS)])
    cfg = dict(vars(cfg), head=EXPLAIN_HEAD, loss=EXPLAIN_LOSS, seed=EXPLAIN_SEED,
               keep_ckpt=True, results_dir="results_explain")
    CKPT = Path(cfg["out_dir"]) / f"{run_name(cfg)}.pt"
    if CKPT.exists():
        print(f"checkpoint موجود است: {CKPT}")
    else:
        run(cfg)
    MAX_LEN = cfg["max_len"]
    %run explain_attention.py --ckpt $CKPT --model $MODEL --head $EXPLAIN_HEAD --max_len $MAX_LEN
else:
    print("skipped (RUN_EXPLAIN=False)")

<div dir="rtl">

### پروزن‌ترین کلمات هر کلاس

اگر کلمات بالای هر کلاس از نظر معنایی مرتبط باشند (مثلاً `fire`، `smoke` و `engine room` برای آتش‌سوزی)، این شاهدی است که مدل به محتوای درست نگاه می‌کند.
اگر کلمات بی‌ربط یا نام کشتی‌ها بالا آمده باشند، ممکن است مدل یک میان‌بر کاذب (spurious shortcut) یاد گرفته باشد.

</div>

In [ ]:
tw_file = Path("reports/top_words_per_class.csv")
if tw_file.exists():
    tw = pd.read_csv(tw_file)
    display(tw[tw["rank"] <= 8].pivot(index="rank", columns="class", values="word"))

<div dir="rtl">

### نمونه‌های رنگ‌آمیزی‌شده

هرچه رنگ قرمزتر باشد، وزن توجه آن کلمه بیشتر است. ✓ یعنی پیش‌بینی درست و ✗ یعنی پیش‌بینی غلط.
نمونه‌های ✗ برای تحلیل خطا ارزشمندترند.

</div>

In [ ]:
ex_file = Path("reports/attention_examples.html")
if ex_file.exists():
    display(HTML(ex_file.read_text()))

<div dir="rtl">

## خروجی‌های نهایی برای پایان‌نامه

| فایل | کاربرد |
|---|---|
| `splits/data_stats.json` | جدول آمار داده (فصل ۳) |
| `reports/summary.md` | جدول اصلی نتایج (فصل ۴) |
| `reports/per_class_f1.csv` و `per_class_recall.csv` | جدول per-class |
| `reports/significance.csv` | آزمون معناداری |
| `reports/top_words_per_class.csv` و `attention_examples.html` | بخش تفسیرپذیری |
| نمودارهای این نوت‌بوک | راست‌کلیک و Save image |

</div>